# Viewing Plain Product

One can use the open-source `radar-data` module to read in the data, convert the base products Z, V, W, D, P, and R into a (6 x Rays x Gates) array and initialize a `radarkit.sweep.Sweep` object. Then, the plotting routine `radarkit.chart.ChartPPI` or `radarkit.chart.ChartRHI` can be used to generate a 6-panel chart.

In [1]:
import os
import blib
import glob
import radar
import numpy as np

blib.useTheme('dark')

import src.radarkit as radarkit
import src.radarkit.chart

In [ ]:
# files = sorted(glob.glob(os.path.expanduser('~/Downloads/data/moment/20231004/*-Z.nc')))
files = sorted(glob.glob(os.path.expanduser('~/Downloads/data/PX*.txz')))
assert len(files) > 0, "No files found"
file = files[min(len(files) - 1, 7)]
print(f"Selected file {file} ({len(files)} files)")

In [3]:
m = radar.re_3parts.match(os.path.basename(file)).groupdict()
scantype = "ppi" if m["scan"][0] == "E" else "rhi" if m["scan"][0] == "A" else "unknown"

In [ ]:
data = radar.read(file)
if np.nanmax(data["products"]["P"]) < 3.142:
    print("Converting P to degrees ...")
    data["products"]["P"] = np.rad2deg(data["products"]["P"])
array = np.stack([data['products'][key].filled(np.nan) for key in ['Z', 'V', 'W', 'D', 'P', 'R']], axis=0)
sweep = radarkit.sweep.Sweep(array,
                             type=scantype,
                             time=data["time"],
                             latitude=data["latitude"],
                             longitude=data["longitude"],
                             e=data["elevations"],
                             a=data["azimuths"],
                             r=data["ranges"],
                             )
# sweep = radarkit.sweep.Sweep(array, type=scantype, de=0.5, dr=20)

In [ ]:
chart = radarkit.chart.ChartRHI(sweep) if scantype == "rhi" else radarkit.chart.ChartPPI(sweep)